# EDA and Silver Preparation

Description: Silver Layer Data Preparation Pipeline. This notebook consolidates Inpatient and Outpatient claims, joins them with Beneficiary demographics, performs feature engineering, and saves the cleansed dataset to the Silver Layer.


In [0]:
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from pyspark.sql.types import *
from pyspark.sql.functions import *
from functools import reduce
from operator import add
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

warnings.filterwarnings("ignore") # Ignore plot warnings for cleaner output

## Loading Data from Unity Catalog

Read tables using Spark SQL API
Instead of reading raw file paths, we directly query the tables managed by Databricks.

In [0]:
print("Loading tables from Databricks Catalog")

# Load the raw tables from the Bronze layer (Unity Catalog)
inpatient_df = spark.table("workspace.insurance_claim.train_inpatientdata")
outpatient_df = spark.table("workspace.insurance_claim.train_outpatientdata")
beneficiary_df = spark.table("workspace.insurance_claim.train_beneficiarydata")

print(f"Total Inpatient Claims: {inpatient_df.count()}")
print(f"Total Outpatient Claims: {outpatient_df.count()}")
print(f"Total Beneficiaries: {beneficiary_df.count()}\n")

### Inpatient Table

In [0]:
inpatient_df.printSchema()

In [0]:
print(f"Number of inpatient_df columns: {len(inpatient_df.columns)}")

In [0]:
inpatient_df.limit(5).toPandas()

### Outpatient Table

In [0]:
outpatient_df.limit(5).toPandas()

In [0]:
outpatient_df.printSchema()

In [0]:
print(f"Number of outpatient_df columns: {len(outpatient_df.columns)}")

### Beneficiary Table

In [0]:
beneficiary_df.limit(5).toPandas()

In [0]:
display(beneficiary_df)

In [0]:
beneficiary_df.printSchema()

In [0]:
print(f"Number of beneficiary_df columns: {len(beneficiary_df.columns)}")

In [0]:
numeric_cols = [col_name for col_name, dtype in beneficiary_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = beneficiary_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

In [0]:
# Replace "1" with "Male" and "2" with "Female" in "Gender" column
beneficiary_df = (beneficiary_df
    .replace({1: 0, 2: 1}, subset=["Gender"])
    .replace({
        "1": "White", 
        "2": "Black", 
        "3": "Asian", 
        "4": "AmericanIndian_Alaska", 
        "5": "Other"
    }, subset=["Race"])
    .replace({"0": "0", "Y": "1"}, subset=["RenalDiseaseIndicator"])
)

In [0]:
beneficiary_df = (beneficiary_df
    # Transform DOD to DateType ('NA' or others will be automatically null)
    .withColumn("DOD", expr("try_cast(DOD as date)"))
    
    # Transform Gender, Race, State, County to StringType
    .withColumn("Gender", col("Gender").cast(IntegerType()))
    .withColumn("Race", col("Race").cast(StringType()))
    .withColumn("State", col("State").cast(StringType()))
    .withColumn("County", col("County").cast(StringType()))
    .withColumn("RenalDiseaseIndicator", col("RenalDiseaseIndicator").cast(IntegerType()))
)

beneficiary_df.printSchema()

In [0]:
display(beneficiary_df)

In [0]:
numeric_cols = [col_name for col_name, dtype in beneficiary_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = beneficiary_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

In [0]:
# Calculate % of Null values
total_rows = beneficiary_df.count()

missing_percent_df = beneficiary_df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) for c in beneficiary_df.columns
])

missing_percent_df.show(vertical=True)

In [0]:
beneficiary_df = (beneficiary_df
    # Create flags to identify clawback behaviors (negative reimbursement amounts)
    .withColumn("IP_Has_Clawback", when(col("IPAnnualReimbursementAmt") < 0, 1).otherwise(0))
    .withColumn("OP_Has_Clawback", when(col("OPAnnualReimbursementAmt") < 0, 1).otherwise(0))
    
    # Extract the exact clawback amount (convert negative to absolute value to capture the magnitude)
    .withColumn("IP_Clawback_Amt", when(col("IPAnnualReimbursementAmt") < 0, abs(col("IPAnnualReimbursementAmt"))).otherwise(0))
    .withColumn("OP_Clawback_Amt", when(col("OPAnnualReimbursementAmt") < 0, abs(col("OPAnnualReimbursementAmt"))).otherwise(0))
    
    # Clean the original columns (if negative, set to 0 assuming no actual reimbursement was made in this period)
    .withColumn("IPAnnualReimbursementAmt_Clean", when(col("IPAnnualReimbursementAmt") < 0, 0).otherwise(col("IPAnnualReimbursementAmt")))
    .withColumn("OPAnnualReimbursementAmt_Clean", when(col("OPAnnualReimbursementAmt") < 0, 0).otherwise(col("OPAnnualReimbursementAmt")))
)

In [0]:
numeric_cols = [col_name for col_name, dtype in beneficiary_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = beneficiary_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

In [0]:
# List of financial columns where Null/Missing values should be treated as 0
financial_cols = [
    "IPAnnualDeductibleAmt", 
    "OPAnnualDeductibleAmt", 
    "IP_Clawback_Amt", 
    "OP_Clawback_Amt", 
    "IPAnnualReimbursementAmt_Clean", 
    "OPAnnualReimbursementAmt_Clean"
]

# Fill missing financial values with 0
# 0 represents 'no activity' rather than 'unknown data'
for col_name in financial_cols:
    beneficiary_df = beneficiary_df.fillna(0, subset=[col_name])

# Feature Engineering for Date of Death (DOD)
# Create a binary flag 'Is_Deceased' to indicate if a patient has passed away
# 1 = Deceased, 0 = Still living
beneficiary_df = beneficiary_df.withColumn(
    "Is_Deceased", 
    F.when(F.col("DOD").isNotNull(), 1).otherwise(0)
)

In [0]:
# Assuming 'DOB' is in string format like 'YYYY-MM-DD'
# Cast string DOB to DateType
beneficiary_df = beneficiary_df.withColumn("BirthDate_Date", F.to_date(F.col("DOB"), "yyyy-MM-dd"))

# Calculate Age: 
# Calculate the difference in years between today and the birth date.
# If the patient is deceased, you could optionally calculate age at death 
# by using 'DOD' instead of 'current_date()'.
beneficiary_df = beneficiary_df.withColumn(
    "Age", 
    F.floor(F.datediff(F.current_date(), F.col("BirthDate_Date")) / 365.25).cast(IntegerType())
)

# Clean up: Drop the temporary date column
beneficiary_df = beneficiary_df.drop("BirthDate_Date")

In [0]:
beneficiary_df.select("DOB", "DOD", "Age", "Is_Deceased").limit(20).toPandas()

In [0]:
numeric_cols = [col_name for col_name, dtype in beneficiary_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = beneficiary_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

In [0]:
beneficiary_df = (beneficiary_df

    # Calculate Total Reimbursement (IP + OP)
    .withColumn("Total_Reimbursement", 
                col("IPAnnualReimbursementAmt_Clean") + col("OPAnnualReimbursementAmt_Clean"))
    
    # Calculate Total Chronic Conditions 
    # (Assuming condition columns contain 1 for 'Yes' and 2 for 'No', we convert 2 to 0 for summing)
    .withColumn("Total_Chronic_Conds", 
        when(col("ChronicCond_Alzheimer") == 1, 1).otherwise(0) +
        when(col("ChronicCond_Heartfailure") == 1, 1).otherwise(0) +
        when(col("ChronicCond_KidneyDisease") == 1, 1).otherwise(0) +
        when(col("ChronicCond_Cancer") == 1, 1).otherwise(0) +
        when(col("ChronicCond_ObstrPulmonary") == 1, 1).otherwise(0) +
        when(col("ChronicCond_Depression") == 1, 1).otherwise(0) +
        when(col("ChronicCond_Diabetes") == 1, 1).otherwise(0) +
        when(col("ChronicCond_IschemicHeart") == 1, 1).otherwise(0) +
        when(col("ChronicCond_Osteoporasis") == 1, 1).otherwise(0) +
        when(col("ChronicCond_rheumatoidarthritis") == 1, 1).otherwise(0) +
        when(col("ChronicCond_stroke") == 1, 1).otherwise(0)
    )
)

In [0]:
display(beneficiary_df)

In [0]:
beneficiary_df.printSchema()

In [0]:
# Filter and display rows where 'Age' is null
missing_age_df = beneficiary_df.filter(F.col("Age").isNull())

# Show the results (limit to 20 rows to avoid clutter)
missing_age_df.limit(20).toPandas()

# Optional: Count how many rows have null Age
print(f"Total records with missing Age: {missing_age_df.count()}")

In [0]:
# create a copy of the dataframe where "NA" strings are treated as actual NULLs
# use .when().otherwise() to replace "NA" with null for all string columns
df_with_nulls = beneficiary_df
for col in beneficiary_df.columns:
    df_with_nulls = df_with_nulls.withColumn(
        col, 
        F.when(F.col(col) == "NA", None).otherwise(F.col(col))
    )

# perform the Missing Value Evaluation on the new 'df_with_nulls'
missing_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_with_nulls.columns]
missing_pd = df_with_nulls.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ['Column_Name', 'Missing_Count']
missing_pd['Missing_Percentage(%)'] = (missing_pd['Missing_Count'] / total_rows) * 100

# Sort and Display
missing_to_show = missing_pd[missing_pd['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

if not missing_to_show.empty:
    print("Found missing values (including 'NA' replaced as Null):")
    display(missing_to_show)
else:
    print("✅ No missing values (or 'NA' strings) found.")

## Data Preparation & Plotting (EDA)

### Financial Relationships: Scatter Plot & Correlation

**Insight:** Disbursement and withholding tax trends normally align. A sharp increase in disbursements paired with an abnormally low deductible may indicate a billing mistake or fraud.

In [0]:
# Set visualization style
sns.set_theme(style="whitegrid")

# Sample 10% of the data for row-level plots (Scatter, Boxplot, Density) to avoid memory crash
# Adjust fraction=0.1 to a smaller/larger number depending on your dataset size
df_sample = beneficiary_df.sample(withReplacement=False, fraction=0.1, seed=42).toPandas()


# =====================================================================
# Financial Relationships: Scatter Plot & Correlation
# =====================================================================
plt.figure(figsize=(10, 6))
# Using log scale for better visualization of highly skewed reimbursement amounts
sns.scatterplot(data=df_sample, x='IPAnnualDeductibleAmt', y='IPAnnualReimbursementAmt_Clean', alpha=0.5)
plt.title("Relationship between IP Deductible and IP Reimbursement")
plt.xlabel("IP Annual Deductible Amount")
plt.ylabel("IP Annual Reimbursement Amount (Cleaned)")
plt.show()

**Key Insights:**

- Standardized Plan Tiers: Vertical clustering along specific X-axis values indicates standardized deductible structures (e.g., fixed $1K, $2K, $3K plans) chosen by large groups of beneficiaries.

- Adverse Selection: Extreme reimbursement claims (>$80K) are heavily concentrated within low-deductible groups (<$5K). High-risk or chronically ill patients likely opt for plans with lower out-of-pocket limits.

- No Linear Correlation: Deductible amounts do not predict total reimbursement. Payouts are driven by medical necessity rather than the plan's deductible size.

**Areas for Investigation (Anomaly Detection):**

- High Deductible, Low Payout (Bottom-Right): Extreme deductibles (>$10K–$30K) paired with minimal reimbursements are highly unusual. These require audits for potential billing errors or swapped data entries.

- Zero Deductible Outliers (Y-Axis Line): Claims with exactly $0 deductibles but substantial payouts (up to $60K) must be cross-referenced with policy rules to rule out deductible evasion or provider-level fraud

### Correlation Analysis

**Insight:** Analyze the overall impact between variables. For instance, see if the months of coverage have a direct correlation with the disbursement amount.

In [0]:
# Define the list of numeric columns to be included in the correlation analysis
numeric_cols = [
    "Age", 
    "Total_Chronic_Conds", 
    "RenalDiseaseIndicator",
    "IPAnnualDeductibleAmt", 
    "OPAnnualDeductibleAmt", 
    "IP_Clawback_Amt", 
    "OP_Clawback_Amt", 
    "Total_Reimbursement"
]

chronic_cond_cols = [
    "ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_KidneyDisease", 
    "ChronicCond_Cancer", "ChronicCond_ObstrPulmonary", "ChronicCond_Depression", 
    "ChronicCond_Diabetes", "ChronicCond_IschemicHeart", "ChronicCond_Osteoporasis", 
    "ChronicCond_rheumatoidarthritis", "ChronicCond_stroke"
]

numeric_cols.extend(chronic_cond_cols)

# Extract data to Pandas
# Using sample(0.1) is a good practice for memory management in large datasets
corr_data = beneficiary_df.select(*numeric_cols).sample(fraction=0.1, seed=42).toPandas()

# Calculate Correlation Matrix
corr_matrix = corr_data.corr()

# Plotting the Heatmap
plt.figure(figsize=(16, 12))

# Create a mask for the upper triangle (Optional: removes redundant mirror image)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,                
    annot=True,              # Display the correlation coefficient values inside the cells
    fmt=".1f",               # Format the numbers to 1 decimal places
    cmap='coolwarm', 
    vmin=-1, vmax=1, 
    center=0,
    linewidths=0.5, 
    cbar_kws={"shrink": .8}
)

plt.title("Correlation Matrix of Beneficiary Features", fontsize=18, pad=20)
plt.tight_layout()
plt.show()

**Key Insights:**

- Inpatient Claims Drive Total Costs: There is a near-perfect positive correlation (0.97) between Inpatient Annual Reimbursement (IPAnnualReimbursementAmt_Clean) and Total Reimbursement (Total_Reimbursement). This indicates that inpatient care is the overwhelming driver of overall healthcare costs, whereas outpatient care has a much weaker impact (0.40).

- Reimbursements and Deductibles Scale Together: Both Inpatient (0.63) and Outpatient (0.70) reimbursements show strong positive correlations with their respective deductible amounts. This logically aligns with insurance mechanics, where higher total claims correspond to patients hitting higher deductible tiers.

- Chronic Conditions Increase Financial Burden: The number of chronic conditions (Total_Chronic_Conds) has a moderate positive correlation with financial metrics, such as Inpatient Deductibles (0.34) and Total Reimbursement (0.33). This validates the assumption that sicker patients cost more to treat.

- Age Shows No Correlation: Surprisingly, Age has near-zero correlation (0.01 - 0.09) with both reimbursement amounts and chronic conditions within this specific dataset.

**Areas for Investigation (Anomalies & Red Flags):**

- Investigate the "Age" Disconnect: In typical healthcare data, age correlates strongly with chronic conditions and costs. The fact that it doesn't here (0.09 and 0.03 respectively) is counterintuitive. You should investigate if this dataset is pre-filtered for a very narrow age group (e.g., exclusively Medicare patients aged 65+) which would artificially compress the variance and hide the correlation.

- Zero Correlation in Coverage Months: The duration of coverage (NoOfMonths_PartACov and PartBCov) shows virtually no correlation (0.00 - 0.01) with annual reimbursements. Logically, someone covered for 12 months should have a higher chance of accumulating claims than someone covered for 1 month. You should verify if the reimbursement amounts are already annualized or if there are data entry errors in the coverage month columns.

### Demographics vs. Costs: Box Plot

**Insight:** Visualizes expense distribution and outliers by population group to detect any segments with unusually high costs.

In [0]:
# Set the visual style for the plots
sns.set_theme(style="whitegrid")

# =====================================================================
# Revised Box Plot: Excluding $0 Claims
# Insight: Observe cost distributions ONLY for patients who actually had inpatient claims.
# This prevents the "zero-inflated" data from flattening the boxes.
# =====================================================================

# Step 1: Filter the data to include only beneficiaries with actual inpatient claims (> 0)
demo_cols = ["Race", "Gender", "IPAnnualReimbursementAmt_Clean"]
demo_cost_df = (beneficiary_df
                .filter(F.col("IPAnnualReimbursementAmt_Clean") > 0) # Keep only positive reimbursement amounts
                .select(*demo_cols)
                .sample(fraction=0.1, seed=42) # Sample 10% to prevent memory overload
                .toPandas())

plt.figure(figsize=(12, 6))

# Step 2: Create the Box Plot
sns.boxplot(
    data=demo_cost_df, 
    x="Race", 
    y="IPAnnualReimbursementAmt_Clean", 
    hue="Gender",       
    palette="Set2"
)

# Step 3: Apply standard Log Scale to the Y-axis
# Since we removed all $0 values, we can safely use standard 'log' instead of 'symlog'
plt.yscale('log')

plt.title("Distribution of IP Annual Reimbursement by Race and Gender (Excluding $0 Claims)", fontsize=14)
plt.xlabel("Race", fontsize=12)
plt.ylabel("IP Annual Reimbursement Amount (Log Scale)", fontsize=12)
plt.legend(title="Gender", loc='upper left')
plt.tight_layout()
plt.show()

**Overview:**

This updated Box Plot illustrates the distribution of Inpatient (IP) Annual Reimbursement across explicitly labeled demographic groups (Race: White, Black, Asian, Other; Gender: Male, Female). By excluding the $0 claims and utilizing a logarithmic scale on the Y-axis, the chart effectively reveals the true cost distributions of patients who actually required inpatient care.

**Key Insights:**

- Consistent Baseline Costs: Across almost all races and genders, the median inpatient reimbursement (the middle line inside the boxes) is remarkably consistent, hovering near the $10,000 mark (10^4). This implies that the baseline cost of an inpatient admission is relatively uniform, regardless of demographic background.

- Elevated Costs for Black Males: The box representing Black Males (the green box above "Black") is slightly wider and elevated compared to the rest. Its 75th percentile and median are noticeably higher, indicating that when admitted, this specific demographic tends to incur higher inpatient costs on average.

- Dense Outliers in the White Demographic: The White population displays the thickest and most continuous stream of extreme outliers above $100,000 ($10^5). Statistically, this is usually an artifact of the White demographic being the vast majority in standard US healthcare datasets, rather than them having a disproportionately higher rate of extreme medical events.

**Areas for Investigation (Next Steps):**

- Investigate the Black Male Disparity: You should cross-reference the Black Male cohort with the Total_Chronic_Conds feature or specific high-cost disease indicators (like Renal Disease or Heart Failure). This will help determine if their elevated cost distribution is driven by a higher burden of chronic illnesses or late-stage interventions.

- Audit Mega-Claims (>$100k): Isolate the beneficiaries represented by the circles above the 10^5 (100,000) line across all races. Analyze these mega-claims to see if they originate from a specific cluster of Providers (Hospitals) or share the same Diagnosis Codes. This is a standard procedure to detect potential systemic overbilling or institutional fraud.

### Geographic Bar Chart: Mean OP Reimbursement by State
**Insight:** Identify states or districts with abnormally high average disbursements, which typically flag where to begin investigations into potentially fraudulent clinic networks.

In [0]:
# =====================================================================
# Geographic Bar Chart: Mean OP Reimbursement by State
# Insight: Identify states with unusually high average outpatient costs.
# =====================================================================

# Aggregate the data in PySpark first (Highly efficient for Big Data)
# Calculate the mean of OPAnnualReimbursementAmt_Clean grouped by State
state_cost_df = (beneficiary_df
                 .filter(F.col("State").isNotNull()) # Remove null states just in case
                 .groupBy("State")
                 .agg(F.mean("OPAnnualReimbursementAmt_Clean").alias("Mean_OP_Reimbursement"))
                 .orderBy(F.desc("Mean_OP_Reimbursement")) # Sort from highest to lowest mean
                 .toPandas())

state_cost_df.count()

In [0]:
plt.figure(figsize=(16, 7))

# Create a Bar Chart
# Using a color palette that gets darker for higher values to emphasize the high-cost states
sns.barplot(
    data=state_cost_df, 
    x="State", 
    y="Mean_OP_Reimbursement", 
    palette="flare_r" 
)

plt.title("Average OP Annual Reimbursement Amount by State", fontsize=15)
plt.xlabel("State Code", fontsize=12)
plt.ylabel("Mean OP Reimbursement Amount", fontsize=12)

# Rotate X-axis labels if there are many states to prevent overlapping text
plt.xticks(rotation=45, ha='right', fontsize=9) 
plt.tight_layout()
plt.show()

**Overview:**

The bar chart correctly displays the Average Outpatient (OP) Annual Reimbursement Amount aggregated by State Code, sorted from highest to lowest.

**Key Insights:**

- No Extreme Anomalies at the State Level: Unlike the previous box plots, this graph displays a very smooth, gradual decline. There are no massive "jumps" or extreme outliers where one state costs $5,000 while the rest cost $1,000.

- Top Cost Drivers: State Codes 2 and 29 are at the very top, with average outpatient reimbursements nearing the $1,550 - $1,600 mark.

- The Dilution Effect: Because the data is aggregated at the state level, the massive claims from individual fraudulent clinics are likely being "diluted" or averaged out by the thousands of normal clinics in the same state.

**Next Steps for Investigation:**

- Since the state-level view is too broad to pinpoint fraud directly, you should use States 2 and 29 as your starting point. Filter your dataset to include only these two states, and then group the data by Provider (Clinic/Hospital) or County. This drill-down method will expose the specific clinics driving these state averages up.

**Next Steps for Investigation:**

- Since the state-level view is too broad to pinpoint fraud directly. This drill-down method will expose the specific clinics driving these state averages up.

In [0]:
sns.set_theme(style="whitegrid")

# =====================================================================
# Drill-down Analysis: Top 20 Counties in State 2 and 29 by Mean OP Cost
# Insight: Identify specific counties driving the high state-level averages.
# =====================================================================

# Step 1: Filter specifically for State 2 and 29
target_states = [2, 29]

# Step 2: Aggregate data at the County level for these specific states
county_cost_df = (beneficiary_df
                  .filter(F.col("State").isin(target_states))
                  .groupBy("State", "County")
                  .agg(
                      F.mean("OPAnnualReimbursementAmt_Clean").alias("Mean_OP_Reimbursement"),
                      F.count("County").alias("Patient_Count") # Count patients to check sample size
                  )
                  # Filter out counties with too few patients to avoid skewed averages (optional, e.g., > 5 patients)
                  .filter(F.col("Patient_Count") > 5) 
                  .orderBy(F.desc("Mean_OP_Reimbursement"))
                  .limit(20) # Focus on the Top 20 most expensive counties
                  .toPandas())

# Step 3: Create a combined label for the X-axis (e.g., "State 2 - County 15")
county_cost_df['State_County'] = 'S:' + county_cost_df['State'].astype(str) + ' C:' + county_cost_df['County'].astype(str)

# Step 4: Plot the Bar Chart
plt.figure(figsize=(16, 7))

# Create bar plot
barplot = sns.barplot(
    data=county_cost_df, 
    x="State_County", 
    y="Mean_OP_Reimbursement", 
    palette="rocket" # Using a dark-to-light palette to emphasize high costs
)

plt.title("Top 20 Most Expensive Counties in States 2 & 29 (Outpatient Reimbursement)", fontsize=15, pad=15)
plt.xlabel("State and County Code", fontsize=12)
plt.ylabel("Mean OP Reimbursement Amount", fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add patient count text on top of each bar to provide context
for index, row in county_cost_df.iterrows():
    barplot.text(
        index, 
        row.Mean_OP_Reimbursement + 20, # Place text slightly above the bar
        f"n={row.Patient_Count}", 
        color='black', 
        ha="center", 
        fontsize=9
    )

plt.tight_layout()
plt.show()

**Overview:**

The bar chart successfully drills down into States 2 & 29, displaying the top counties by Mean Outpatient (OP) Reimbursement. The addition of the patient count (n) on each bar provides critical context for distinguishing between statistical noise and systemic issues.

**Key Insights:**

- The "Significant Jump" (The Top 4): There is a clear, massive drop-off between the 4th county (S:2 C:110, ~$2,500) and the 5th county (S:29 C:110, ~$1,900). The top 4 counties are exceptionally expensive compared to the rest of the high-cost list.

- Small Sample Size Anomalies (Red Flags): The top 4 most expensive counties (S:2 C:90, S:29 C:20, S:2 C:120, and S:2 C:110) all have very small patient counts (n = 7 to 20). This indicates that their extremely high averages (nearly $3,000) are likely driven by just a few massively inflated claims from a specific provider in that area. This is a classic pattern for targeted overbilling.

- The Systemic High-Cost Area: Look at the 6th bar (S:29 C:10). It has a high average reimbursement of ~$1,600, but it is backed by a massive patient count of n=502. This is not a statistical fluke. It means that high billing is a widespread, systemic practice in this specific county.

**Areas for Investigation (Next Steps):**

- Audit the Top 4 Counties: Filter your dataset for State = 2, County = 90, State = 29, County = 20, etc. Since there are only 7 to 20 patients in these areas, you can easily examine them row-by-row. Identify which Provider ID submitted these claims. It is highly probable you will find a single clinic driving these numbers up.

- Investigate S:29 C:10: Since this county has 502 patients with a consistently high average, you should group the data within this specific county by Provider. This will reveal if the high costs are caused by a network of colluding clinics or if it's just a heavily populated area with a legitimate high-cost specialty hospital.

### Bar Chart: Impact of Renal Disease (ESRD) on Costs

**Insight:** End-Stage Renal Disease (ESRD) patients should expectedly have significantly higher costs than average patients. This chart helps validate that the data is logical and consistent.

In [0]:
# Set the visual style for the plots
sns.set_theme(style="whitegrid")

# =====================================================================
# Bar Chart: Impact of Renal Disease (ESRD) on Costs
# Insight: Verify that ESRD patients (1) have significantly higher costs than normal (0).
# =====================================================================

# Aggregate data in PySpark (Calculate mean IP cost grouped by Renal Disease Indicator)
renal_cost_df = (beneficiary_df
                 .filter(F.col("RenalDiseaseIndicator").isNotNull())
                 .groupBy("RenalDiseaseIndicator")
                 .agg(F.mean("IPAnnualReimbursementAmt_Clean").alias("Mean_IP_Cost"))
                 .orderBy("RenalDiseaseIndicator")
                 .toPandas())

renal_cost_df.count()

In [0]:
# Plot the Bar Chart
plt.figure(figsize=(8, 6))
sns.barplot(
    data=renal_cost_df,
    x="RenalDiseaseIndicator",
    y="Mean_IP_Cost",
    palette="viridis"
)

plt.title("Average IP Reimbursement by Renal Disease Indicator", fontsize=14)
plt.xlabel("Renal Disease Indicator (0 = No, 1 = Yes)", fontsize=12)
plt.ylabel("Mean IP Annual Reimbursement", fontsize=12)
plt.show()

**Overview:**

This bar chart displays the Average Inpatient (IP) Annual Reimbursement Amount, categorized by whether a patient has a Renal Disease Indicator (0 = No, 1 = Yes).

**Key Insights:**

- Significant Cost Disparity: There is a stark difference in costs between the two groups. Patients without renal disease have an average inpatient cost of roughly $3,000, whereas patients with renal disease have an average cost exceeding $7,000 (more than double).

- Data Validation: This chart acts as an excellent "sanity check" for your dataset. End-Stage Renal Disease (ESRD) requires intensive, expensive treatments like dialysis or kidney transplants. Seeing a massive spike in inpatient costs for this specific demographic perfectly aligns with real-world medical economics.

**Areas for Investigation (Next Steps):**

- Look for "Upcoding" Fraud: Since renal disease justifies significantly higher payouts, corrupt providers might falsely flag healthy patients with this indicator to claim more money. You should isolate patients with a "Yes" indicator who never have procedure codes related to dialysis or kidney treatment in their claims.

- Analyze the "No" Outliers: Filter the "No" group and look for patients who have reimbursement amounts well over the $7,000 average of the sick group. What is driving their massive costs if it isn't renal disease?

### Feature Engineering & Density Plot (KDE): Complexity of Chronic Conditions

**Insight:** Multiple co-occurring chronic conditions should trigger an exponential increase in patient costs.

In [0]:
# Set the visual style for the plots
sns.set_theme(style="whitegrid")

# =====================================================================
# Feature Engineering & Density Plot (KDE): Complexity of Chronic Conditions
# Insight: Costs should increase exponentially as chronic conditions compound.
# =====================================================================

# Feature Engineering - Create Total_Chronic_Conds column
# Note: Medicare data usually uses 1 for 'Yes' and 2 for 'No'. 
# We convert 1 to 1, and everything else to 0, then sum them up.
chronic_cols = [
    "ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_KidneyDisease",
    "ChronicCond_Cancer", "ChronicCond_ObstrPulmonary", "ChronicCond_Depression",
    "ChronicCond_Diabetes", "ChronicCond_IschemicHeart", "ChronicCond_Osteoporasis",
    "ChronicCond_rheumatoidarthritis", "ChronicCond_stroke"
]

# Create an expression that dynamically sums all condition columns
conditions_list = [F.when(F.col(c) == 1, 1).otherwise(0) for c in chronic_cols]
sum_expression = reduce(add, conditions_list)

beneficiary_df = beneficiary_df.withColumn("Total_Chronic_Conds", sum_expression)


# Sample the data for the KDE plot (to avoid memory crashes with big data)
kde_df = (beneficiary_df
          .select("Total_Chronic_Conds", "IPAnnualReimbursementAmt_Clean")
          .sample(fraction=0.1, seed=42)
          .toPandas())

# Group the number of conditions for a cleaner KDE plot
# Plotting 11 overlapping lines is messy, so we group them into Low, Medium, and High burden.
kde_df['Condition_Burden'] = np.where(kde_df['Total_Chronic_Conds'] <= 3, 'Low (0-3)',
                             np.where(kde_df['Total_Chronic_Conds'] <= 7, 'Medium (4-7)', 
                                      'High (8-11)'))

# Sort the categories so the legend appears in the correct order
kde_df['Condition_Burden'] = pd.Categorical(
    kde_df['Condition_Burden'], 
    categories=['Low (0-3)', 'Medium (4-7)', 'High (8-11)'], 
    ordered=True
)

# Plot the KDE Plot
plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=kde_df,
    x="IPAnnualReimbursementAmt_Clean",
    hue="Condition_Burden",
    fill=True,              # Fill the area under the curve
    common_norm=False,      # Normalize each curve independently to see the shape clearly
    palette="flare",
    alpha=0.4
)

plt.title("Density of IP Costs by Chronic Condition Burden", fontsize=15)
plt.xlabel("IP Annual Reimbursement Amount", fontsize=12)
plt.ylabel("Density", fontsize=12)

# Limit the X-axis to ignore extreme outliers and focus on the main distribution shift
plt.xlim(0, 60000) 
plt.tight_layout()
plt.show()

**Overview:**

This is a Kernel Density Estimate (KDE) plot showing the probability distribution of Inpatient (IP) Annual Reimbursement Amounts. The data is segmented into three cohorts based on their Condition_Burden (the number of chronic conditions a patient has: Low, Medium, and High).

**Key Insights:**

- Low Burden = Low Cost Concentration: The "Low (0-3)" group (light orange) has a massive, sharp peak near $0. This indicates that the vast majority of relatively healthy patients incur little to no inpatient costs. Their distribution drops off steeply, meaning very few have high claims.

- High Burden = Flattened Distribution & High Costs: The "High (8-11)" group (purple) shows a completely different shape. The peak near $0 is much lower, and the curve is much wider, stretching far to the right (the "fat tail"). This signifies a high variance in costs; patients with 8-11 chronic conditions are highly likely to incur significant inpatient expenses ranging from $10,000 to over $40,000.

- The Shift (Medium Burden): The "Medium (4-7)" group (pink) acts as a perfect transition state, showing a lower peak than the Low group but a shorter tail than the High group.

**Areas for Investigation:**

- Low Burden Outliers: Identify patients in the "Low (0-3)" burden group who appear in the high-cost tail (>$20,000). These cases warrant an audit to determine if they are legitimate medical emergencies or potential upcoding/billing errors.

- Provider Benchmarking: Group the "High (8-11)" burden patients by Provider ID. If certain providers consistently report costs significantly above the curve for this high-burden group, it may indicate excessive billing practices.

### Clawback & Anomalies: Pie Chart & Histogram

**Insight:** Monitors the percentage of patients with clawed-back claims. A disproportionately high rate could signal flaws in the initial billing and approval system.

In [0]:
# Set the visual style for the plots
sns.set_theme(style="whitegrid")

# Create a figure with two subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# =====================================================================
# Pie Chart: Proportion of IP Clawbacks
# Insight: Check if the percentage of patients with clawbacks is unusually high.
# If the proportion is abnormally large, it may indicate systemic claim issues.
# =====================================================================

# Aggregate data in PySpark (Highly efficient for big data)
clawback_counts_df = (beneficiary_df
                      .groupBy("IP_Has_Clawback")
                      .count()
                      .orderBy("IP_Has_Clawback") # Ensure consistent order: 0 then 1
                      .toPandas())

# Define labels and colors for the pie chart
labels = ['No Clawback', 'Has Clawback']
colors = ['#66b3ff', '#ff9999']

# Add function to manage number in Pie Chart (Show both number and percentage)
def make_autopct(values):
    def my_autopct(pct):
        total = values.sum()
        val = int(pct * total / 100.0 + 0.5)
        # define 3 decimal and following by number of patient
        return '{p:.3f}%\n({v:d})'.format(p=pct, v=val)
    return my_autopct



# Plot the Pie Chart on the first subplot (axes[0])
axes[0].pie(
    clawback_counts_df['count'], 
    labels=labels, 
    autopct=make_autopct(clawback_counts_df['count']),
    colors=colors, 
    startangle=90,           # Start the first slice at 90 degrees
    explode=(0, 0.1)         # Slightly pull out the 'Has Clawback' slice for emphasis
)
axes[0].set_title("Proportion of Beneficiaries with IP Clawbacks", fontsize=14, pad=15)


# =====================================================================
# Histogram (Log Scale): Distribution of IP Clawback Amounts
# Insight: Determine if clawbacks are mostly small (clerical/paperwork errors) 
# or massive (potential intentional overbilling / fraud).
# =====================================================================

# Filter only beneficiaries who have a clawback amount > 0
# Note: If this filtered dataset is still massive, you can append .sample(fraction=0.1) 
# before .toPandas() to prevent memory overload.
clawback_amt_df = (beneficiary_df
                   .filter(F.col("IP_Clawback_Amt") > 0)
                   .select("IP_Clawback_Amt")
                   .toPandas())


# Plot the Histogram on the second subplot (axes[1])
# Using log_scale=True handles the extreme range of clawback amounts perfectly
sns.histplot(
    data=clawback_amt_df, 
    x="IP_Clawback_Amt", 
    bins=30,                 # Number of bins
    ax=axes[1], 
    color='salmon',
    log_scale=True           # Automatically applies Log Scale to the X-axis
)

axes[1].set_title("Distribution of IP Clawback Amounts (Log Scale)", fontsize=14, pad=15)
axes[1].set_xlabel("Clawback Amount (Log Scale)", fontsize=12)
axes[1].set_ylabel("Number of Beneficiaries", fontsize=12)

# Adjust layout to prevent overlap and display the plots
plt.tight_layout()
plt.show()

**Overview:**

The analysis utilizes a pie chart to assess the frequency of IP clawbacks and a histogram to evaluate the magnitude of these clawback amounts.

**Key Insights:**

- IP clawbacks are extremely rare, affecting only 15 beneficiaries (0.011%) out of 138,541 total patients, indicating that the overall billing system is generally operating normally.

- The histogram reveals that most clawback amounts are small (ranging from $10 to $1,000), which are typically attributed to clerical errors or duplicate billing.

- A significant red flag exists in the form of a single patient with a clawback amount nearing $10,000, which represents a potential anomaly requiring further investigation.

**Areas for Investigation (Next Steps):**

- Focus data investigation on the 15 identified beneficiaries, particularly the individual with the highest clawback amount.

- Examine the specific providers (hospitals or clinics) associated with these clawback cases to determine if fraud originated from the facility level.

- Analyze the ordering physicians and the diagnosis codes used for these claims to identify patterns of intentional overbilling.

# Combine Inpatient and Outpatient Claims

Identify and merge different claim types
Adding a flag to distinguish between Inpatient and Outpatient claims before merging them into a single comprehensive claims table.


Inpatient data has AdmissionDt (Admission Date) column while Outpatient data is not

In [0]:
# Disable ANSI mode to allow graceful casting of 'NA' strings to Null without throwing errors
spark.conf.set("spark.sql.ansi.enabled", "false")

print("Consolidating Inpatient and Outpatient Claims")

# Add a 'ClaimType' identifier to distinguish claim sources
inpatient_df = inpatient_df.withColumn("ClaimType", F.lit("Inpatient"))
outpatient_df = outpatient_df.withColumn("ClaimType", F.lit("Outpatient"))

# Union the dataframes
# allowMissingColumns=True is crucial here because Inpatient data contains
# admission/discharge dates that Outpatient data does not have.
all_claims_df = inpatient_df.unionByName(outpatient_df, allowMissingColumns=True)

print(f"Total Consolidated Claims before join: {all_claims_df.count()}\n")

# Display the dataframe to explore the Data Profile feature
display(all_claims_df)

In [0]:
all_claims_df.printSchema()

In [0]:
# Check missing values
total_rows = all_claims_df.count()

missing_percent_df = all_claims_df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) for c in all_claims_df.columns
])

missing_percent_df.show(vertical=True)

In [0]:
numeric_cols = [col_name for col_name, dtype in all_claims_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = all_claims_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

### Data Cleaning (Pre-Join)
**Goal:** Ensure the 'all_claims_df' is clean before joining with demographics.

In [0]:
# Remove duplicate claims based on ClaimID
# This prevents row explosion during the Join operation
all_claims_cleaned = all_claims_df.dropDuplicates(["ClaimID"])

# Filter out temporal logic errors (Discharge before Admission)
all_claims_cleaned = all_claims_cleaned.filter(
    (F.col("ClaimEndDt") >= F.col("ClaimStartDt")) | F.col("ClaimEndDt").isNull()
)

# Clean financial values (Ensure no negative amounts)
all_claims_cleaned = all_claims_cleaned.filter(
    (F.col("InscClaimAmtReimbursed") >= 0) | F.col("InscClaimAmtReimbursed").isNull()
)

print(f"Original count: {all_claims_df.count():,}")
print(f"Cleaned count: {all_claims_cleaned.count():,}")

# Now, you can safely perform the Join using 'all_claims_cleaned'
# cleaned_df = all_claims_cleaned.join(beneficiary_df, on="BeneID", how="left")

In [0]:
# =============================================================================
# DATASET OVERVIEW & QUALITY CHECKS
# =============================================================================

print("Fixing data types (Casting financial strings to integers)...")
all_claims_df = all_claims_df.withColumn("DeductibleAmtPaid", F.col("DeductibleAmtPaid").cast(IntegerType()))

# Dataset Dimensions
total_rows = all_claims_df.count()
total_cols = len(all_claims_df.columns)
print(f"Total Records: {total_rows:,} | Total Features: {total_cols}")

# Missing Values (Nulls) Evaluation
print("\nMissing Values Evaluation")
missing_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in all_claims_df.columns]

# Calculate nulls, convert to Pandas, and reset index to turn the column name into a real column
missing_pd = all_claims_df.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ['Column_Name', 'Missing_Count'] # Renaming the columns clearly

# Calculate percentage
missing_pd['Missing_Percentage(%)'] = (missing_pd['Missing_Count'] / total_rows) * 100

# Display only columns that have missing values
missing_to_show = missing_pd[missing_pd['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

if not missing_to_show.empty:
    print("Found columns with missing values:")
    display(missing_to_show)
else:
    print("✅ No missing values found in the dataset.")

In [0]:
# Filter and display rows where 'DeductibleAmtPaid' is null
missing_to_show = all_claims_df.filter(F.col("DeductibleAmtPaid").isNull())

# Show the results (limit to 10 rows to avoid clutter)
missing_to_show.limit(10).toPandas()

In [0]:
# create a copy of the dataframe where "NA" strings are treated as actual NULLs
# use .when().otherwise() to replace "NA" with null for all string columns
df_with_nulls = all_claims_df
for col in all_claims_df.columns:
    df_with_nulls = df_with_nulls.withColumn(
        col, 
        F.when(F.col(col) == "NA", None).otherwise(F.col(col))
    )

# perform the Missing Value Evaluation on the new 'df_with_nulls'
missing_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_with_nulls.columns]
missing_pd = df_with_nulls.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ['Column_Name', 'Missing_Count']
missing_pd['Missing_Percentage(%)'] = (missing_pd['Missing_Count'] / total_rows) * 100

# Sort and Display
missing_to_show = missing_pd[missing_pd['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

if not missing_to_show.empty:
    print("Found missing values (including 'NA' replaced as Null):")
    display(missing_to_show)
else:
    print("✅ No missing values (or 'NA' strings) found.")

### EDA: Smart Correlation Heatmap (Handling String-Numeric Data)

**Goal:** Automatically detect numerical patterns within string-type columns and visualize correlations.

In [0]:
# Sample a subset of the data for performance optimization
# Drawing a 10% sample is sufficient for statistical trends and prevents memory overflow
sample_df = all_claims_df.sample(fraction=0.1, seed=42).toPandas()


# Force conversion of string columns to numeric
# Many financial columns are imported as strings due to 'NA' values.
# 'errors="coerce"' turns non-convertible strings (like names) into NaN.
print("\n--- Force Converting Strings to Numeric ---")
numeric_features = []

for col in sample_df.columns:
    # Attempt to convert column to numeric
    converted_col = pd.to_numeric(sample_df[col], errors='coerce')
    
    # Check if the column contains enough valid numeric data (at least 10% valid)
    # This filters out columns that are truly categorical/text-based.
    valid_number_count = converted_col.notna().sum()
    if valid_number_count > (len(sample_df) * 0.1):
        sample_df[col] = converted_col
        numeric_features.append(col)

print(f"✅ Found {len(numeric_features)} columns containing valid numerical data:")
print(numeric_features)



# Plotting the correlation heatmap
# Using a heatmap helps identify redundant features (multicollinearity)
print("\n--- Plotting Correlation Heatmap ---")
if len(numeric_features) > 1:
    # Compute the correlation matrix
    corr_matrix = sample_df[numeric_features].corr()

    # Create the heatmap with a mask to show only the lower triangle (for better readability)
    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(
        corr_matrix, 
        annot=True,              # Show correlation coefficients
        mask=mask,               # Hide upper triangle
        cmap='coolwarm',         # Color scheme: Red (positive) to Blue (negative)
        fmt=".2f",               # Two decimal places
        vmin=-1, vmax=1, 
        linewidths=0.5,
        cbar_kws={"shrink": .8}
    )
    plt.title('Correlation Heatmap of Detected Numerical Features', fontsize=16, pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns found to create a correlation heatmap.")

### Feature Engineering: Financial Ratios for Fraud Detectionฃ

**1. TotalClaimCost (Total Medical Cost)**

- Feature Name: TotalClaimCost

- Logic: DeductibleAmtPaid + InscClaimAmtReimbursed

- Reasoning: This represents the actual total medical cost of the treatment. It serves as a key indicator for detecting whether a specific claim value is abnormally high.

**2. DeductibleRatio (Responsibility Ratio)**

- Feature Name: DeductibleRatio

- Logic: DeductibleAmtPaid / (TotalClaimCost + 1) (Note: Adding 1 prevents a division-by-zero error).

- Reasoning: If a claim shows an abnormally low deductible compared to the insurance reimbursement amount (high reimbursement but near-zero deductible), it may signal potential fraud, where documents are falsified so that insurance covers the full amount without any patient out-of-pocket payment.

In [0]:
# ==============================================================================
# Feature Engineering: Financial Ratios for Fraud Detection
# ==============================================================================

# Create 'TotalClaimCost'
# Capturing the total financial impact of the claim
all_claims_df = all_claims_df.withColumn(
    "TotalClaimCost", 
    F.col("InscClaimAmtReimbursed") + F.col("DeductibleAmtPaid")
)

# Create 'DeductibleRatio'
# Identifying anomalous financial structures between patient and insurance payment
all_claims_df = all_claims_df.withColumn(
    "DeductibleRatio", 
    F.round(F.col("DeductibleAmtPaid") / (F.col("TotalClaimCost") + 1), 2)
)

# Define your financial columns
financial_cols = ["InscClaimAmtReimbursed", "DeductibleAmtPaid", "TotalClaimCost", "DeductibleRatio"]

# Apply casting to each column
for col_name in financial_cols:
    if col_name in all_claims_df.columns:
        all_claims_df = all_claims_df.withColumn(col_name, F.col(col_name).cast(DoubleType()))


print("Engineered new financial features: TotalClaimCost and DeductibleRatio.")

# Preview the new features
display(all_claims_df.select("ClaimID", "InscClaimAmtReimbursed", "DeductibleAmtPaid", "TotalClaimCost", "DeductibleRatio").limit(5))

In [0]:
all_claims_df.printSchema()

In [0]:
# Logical Errors & Anomalies
print("\n -- Anomaly & Business Logic Checks --")
# Duplicates
dup_count = all_claims_df.groupBy("ClaimID").count().filter(F.col("count") > 1).count()
print(f"Duplicate ClaimIDs found: {dup_count}")

# Time Travel (End Date < Start Date)
temp_error_count = all_claims_df.filter(F.col("ClaimEndDt") < F.col("ClaimStartDt")).count()
print(f"Temporal errors (Discharge before Admission): {temp_error_count}")

# Negative Financials
neg_financial_count = all_claims_df.filter((F.col("InscClaimAmtReimbursed") < 0) | (F.col("DeductibleAmtPaid") < 0)).count()
print(f"Negative financial amounts found: {neg_financial_count}")

In [0]:
# Sample a subset of the data for performance optimization
# Drawing a 10% sample is sufficient for statistical trends and prevents memory overflow
sample_df = all_claims_df.sample(fraction=0.1, seed=42).toPandas()


# Force conversion of string columns to numeric
# Many financial columns are imported as strings due to 'NA' values.
# 'errors="coerce"' turns non-convertible strings (like names) into NaN.
print("\n--- Force Converting Strings to Numeric ---")
numeric_features = []

for col in sample_df.columns:
    # Attempt to convert column to numeric
    converted_col = pd.to_numeric(sample_df[col], errors='coerce')
    
    # Check if the column contains enough valid numeric data (at least 10% valid)
    # This filters out columns that are truly categorical/text-based.
    valid_number_count = converted_col.notna().sum()
    if valid_number_count > (len(sample_df) * 0.1):
        sample_df[col] = converted_col
        numeric_features.append(col)

print(f"✅ Found {len(numeric_features)} columns containing valid numerical data:")
print(numeric_features)



# Plotting the correlation heatmap
# Using a heatmap helps identify redundant features (multicollinearity)
print("\n--- Plotting Correlation Heatmap ---")
if len(numeric_features) > 1:
    # Compute the correlation matrix
    corr_matrix = sample_df[numeric_features].corr()

    # Create the heatmap with a mask to show only the lower triangle (for better readability)
    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(
        corr_matrix, 
        annot=True,              # Show correlation coefficients
        mask=mask,               # Hide upper triangle
        cmap='coolwarm',         # Color scheme: Red (positive) to Blue (negative)
        fmt=".2f",               # Two decimal places
        vmin=-1, vmax=1, 
        linewidths=0.5,
        cbar_kws={"shrink": .8}
    )
    plt.title('Correlation Heatmap of Detected Numerical Features', fontsize=16, pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns found to create a correlation heatmap.")

### Missing Value Imputation based on Medical Domain Logic

In [0]:
# =============================================================================
# Missing Value Imputation based on Medical Domain Logic
# =============================================================================

# Automatically identify all diagnosis and procedure code columns
# This covers ClmProcedureCode_1 to 6, ClmDiagnosisCode_1 to 10, and ClmAdmitDiagnosisCode
diag_cols = [c for c in all_claims_df.columns if "ClmDiagnosisCode" in c or "ClmAdmitDiagnosisCode" in c]
proc_cols = [c for c in all_claims_df.columns if "ClmProcedureCode" in c]

# Create a dictionary for imputation
# Fill with "None" for diagnosis and procedure codes (indicating no such disease/procedure was recorded)
fillna_dict = {c: "None" for c in (diag_cols + proc_cols)}

# Impute specific values for categorical columns based on domain knowledge
fillna_dict.update({
    "OperatingPhysician": "None",         # Indicates no operating physician was involved
    "DiagnosisGroupCode": "Outpatient"    # Missing group code indicates an outpatient claim
})

# Apply the imputation to the DataFrame all at once for better performance
all_claims_df = all_claims_df.fillna(fillna_dict)

print("✅ Imputed missing values correctly based on domain logic.")

# Verify the results to ensure missing values are handled (checking a few specific columns)
all_claims_df.select(
    "ClmProcedureCode_6", "ClmDiagnosisCode_10", "AdmissionDt", "DiagnosisGroupCode", "OperatingPhysician"
).limit(5).toPandas()

In [0]:
# =============================================================================
# Advanced Replacement: Handling "NA" strings that .fillna() missed
# =============================================================================


# Define columns that need "NA" replacement
# These are the ones we identified earlier
target_cols = [c for c in all_claims_df.columns if "ClmDiagnosisCode" in c or "ClmProcedureCode" in c]
target_cols += ["OperatingPhysician", "DiagnosisGroupCode"]

# Iterate through columns and replace "NA" with "None" or "Outpatient"
for col_name in target_cols:
    if col_name == "DiagnosisGroupCode":
        # Replace "NA" with "Outpatient" specifically for this column
        all_claims_df = all_claims_df.withColumn(
            col_name, 
            F.when(F.col(col_name) == "NA", "Outpatient").otherwise(F.col(col_name))
        )
    else:
        # Replace "NA" with "None" for all other ID columns
        all_claims_df = all_claims_df.withColumn(
            col_name, 
            F.when(F.col(col_name) == "NA", "None").otherwise(F.col(col_name))
        )

print("✅ Successfully replaced 'NA' strings with domain-appropriate values.")

# Verify the result again
all_claims_df.select(
    "ClmProcedureCode_6", "ClmDiagnosisCode_10", "AdmissionDt", "DiagnosisGroupCode", "OperatingPhysician"
).limit(5).toPandas()

### Verification: Scan for any remaining "NA" strings

In [0]:
# =============================================================================
# Verification: Scan for any remaining "NA" strings
# =============================================================================

target_cols = [c for c in all_claims_df.columns if "ClmDiagnosisCode" in c or "ClmProcedureCode" in c]
target_cols += ["OperatingPhysician", "DiagnosisGroupCode", "AdmissionDt", "DischargeDt"]

# Loop through all target columns and count if "NA" still exists
print("Scanning for any remaining 'NA' strings...")
for col_name in target_cols:
    na_count = all_claims_df.filter(F.col(col_name) == "NA").count()
    if na_count > 0:
        print(f"⚠️ Warning: Found {na_count} 'NA' values in column: {col_name}")
    else:
        print(f"✅ Clean: No 'NA' in {col_name}")

In [0]:
# Check if there are still any Nulls in the cleaned columns
# If we successfully replaced Nulls with "None", the count should be 0
null_check_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in target_cols]
all_claims_df.select(*null_check_exprs).toPandas()

**Create a new feature 'Is_Inpatient'**

In [0]:
# Create a new feature 'Is_Inpatient'
# If AdmissionDt is not null, it's an Inpatient (1). Otherwise, it's an Outpatient (0).
all_claims_df = all_claims_df.withColumn(
    "Is_Inpatient", 
    F.when(F.col("AdmissionDt").isNotNull(), 1).otherwise(0)
)

display(all_claims_df)

**Calculate Length of Stay safely**

In [0]:
all_claims_df = all_claims_df.withColumn(
    "LengthOfStay",
    F.when(F.col("Is_Inpatient") == 1, 
           F.datediff(F.to_date(F.col("DischargeDt")), F.to_date(F.col("AdmissionDt"))))
     .otherwise(0)
)

display(all_claims_df)

In [0]:
numeric_cols = [col_name for col_name, dtype in all_claims_df.dtypes if dtype in ('bigint', 'int', 'double')]

summary_df = all_claims_df.select(*numeric_cols).describe().toPandas()

display(summary_df)

In [0]:
# Sample a subset of the data for performance optimization
# Drawing a 10% sample is sufficient for statistical trends and prevents memory overflow
sample_df = all_claims_df.sample(fraction=0.1, seed=42).toPandas()


# Force conversion of string columns to numeric
# Many financial columns are imported as strings due to 'NA' values.
# 'errors="coerce"' turns non-convertible strings (like names) into NaN.
print("\n--- Force Converting Strings to Numeric ---")
numeric_features = []

for col in sample_df.columns:
    # Attempt to convert column to numeric
    converted_col = pd.to_numeric(sample_df[col], errors='coerce')
    
    # Check if the column contains enough valid numeric data (at least 10% valid)
    # This filters out columns that are truly categorical/text-based.
    valid_number_count = converted_col.notna().sum()
    if valid_number_count > (len(sample_df) * 0.1):
        sample_df[col] = converted_col
        numeric_features.append(col)

print(f"✅ Found {len(numeric_features)} columns containing valid numerical data:")
print(numeric_features)



# Plotting the correlation heatmap
# Using a heatmap helps identify redundant features (multicollinearity)
print("\n--- Plotting Correlation Heatmap ---")
if len(numeric_features) > 1:
    # Compute the correlation matrix
    corr_matrix = sample_df[numeric_features].corr()

    # Create the heatmap with a mask to show only the lower triangle (for better readability)
    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(
        corr_matrix, 
        annot=True,              # Show correlation coefficients
        mask=mask,               # Hide upper triangle
        cmap='coolwarm',         # Color scheme: Red (positive) to Blue (negative)
        fmt=".2f",               # Two decimal places
        vmin=-1, vmax=1, 
        linewidths=0.5,
        cbar_kws={"shrink": .8}
    )
    plt.title('Correlation Heatmap of Detected Numerical Features', fontsize=16, pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns found to create a correlation heatmap.")

### Feature Selection (Handling Multicollinearity)

Goal: Remove redundant financial columns that cause 1.00 correlation

In [0]:
# =============================================================================
# Feature Selection (Handling Multicollinearity)
# =============================================================================

# Define the redundant columns to drop based on the correlation heatmap
# - InscClaimAmtReimbursed (Corr 1.00 with TotalClaimCost)
# - DeductibleAmtPaid (Corr 0.70 with TotalClaimCost, and its behavior is captured in DeductibleRatio)
redundant_cols = [
    "InscClaimAmtReimbursed",
    "DeductibleAmtPaid"
]

# Drop the columns from the dataframe
all_claims_cleaned_df = all_claims_df.drop(*redundant_cols)

print(f"✅ Successfully dropped redundant features: {redundant_cols}")

# Verify the remaining columns to ensure our engineered features are still there
print("\nCurrent columns in all_claims_df:")
print(all_claims_cleaned_df.columns)

In [0]:
display(all_claims_df)

## Data Integration (Joining Claims with Beneficiary Data)

Goal: Combine cleaned claims data with patient demographics using BeneID

In [0]:
# =============================================================================
# Data Integration (Joining Claims with Beneficiary Data)
# =============================================================================

# Perform a Left Join using 'BeneID' as the key
# Using 'left' ensures that all valid claims are kept, even if demographic data is missing
final_joined_df = all_claims_cleaned_df.join(beneficiary_df, on="BeneID", how="left")

print("✅ Successfully joined claims data with beneficiary data.")

In [0]:
# =============================================================================
# Post-Join Quality Check
# =============================================================================

# Check dataset dimensions after join
final_rows = final_joined_df.count()
final_cols = len(final_joined_df.columns)
print(f"Total Records: {final_rows:,} | Total Features: {final_cols}")

# Verify if the join introduced any unexpected missing values in demographic columns
# (e.g., checking if any claim didn't find a matching beneficiary)
missing_bene_count = final_joined_df.filter(F.col("Gender").isNull()).count()

if missing_bene_count > 0:
    print(f"⚠️ Warning: Found {missing_bene_count} claims without matching beneficiary data.")
    # Optional: Fill missing demographics with placeholders if necessary
    final_joined_df = final_joined_df.fillna({"Gender": 0, "Race": 0})
else:
    print("✅ Perfect Match: All claims have corresponding beneficiary data.")

In [0]:
# Preview the final dataset
final_joined_df.select("BeneID", "ClaimID", "TotalClaimCost", "Is_Inpatient", "Gender").limit(5).toPandas()

### Join Train table

In [0]:
print("Loading tables from Databricks Catalog")

# Load the raw tables from the Bronze layer (Unity Catalog)
train_df = spark.table("workspace.insurance_claim.train_claims")

print(f"Total Train Data: {train_df.count()}")

In [0]:
train_df.printSchema()

In [0]:
display(train_df)

In [0]:
# Check missing values
total_train_rows = train_df.count()

missing_percent_df = train_df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_train_rows * 100).alias(c) for c in train_df.columns
])

missing_percent_df.show(vertical=True)

In [0]:
# create a copy of the dataframe where "NA" strings are treated as actual NULLs
# use .when().otherwise() to replace "NA" with null for all string columns
df_with_nulls = train_df
for col in train_df.columns:
    df_with_nulls = df_with_nulls.withColumn(
        col, 
        F.when(F.col(col) == "NA", None).otherwise(F.col(col))
    )

# perform the Missing Value Evaluation on the new 'df_with_nulls'
missing_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_with_nulls.columns]
missing_pd = df_with_nulls.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ['Column_Name', 'Missing_Count']
missing_pd['Missing_Percentage(%)'] = (missing_pd['Missing_Count'] / total_rows) * 100

# Sort and Display
missing_to_show = missing_pd[missing_pd['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

if not missing_to_show.empty:
    print("Found missing values (including 'NA' replaced as Null):")
    display(missing_to_show)
else:
    print("✅ No missing values (or 'NA' strings) found.")

In [0]:
# =============================================================================
# Clean and Prepare the Target Variable (Train Table)
# =============================================================================

print("--- Preparing Target Label ---")

# Convert 'Yes'/'No' string to 1/0 Integer (Standard ML format)
train_df = train_df.withColumn(
    "is_fraud",
    F.when(F.col("PotentialFraud") == "Yes", 1).otherwise(0)
)

# Drop the old string column to keep it clean
train_df = train_df.drop("PotentialFraud")

print("✅ Target variable 'PotentialFraud' encoded to 'is_fraud' (1/0).")
train_df.limit(5).toPandas()

In [0]:
# =============================================================================
# Join Target Labels with Main Dataset
# =============================================================================

print("--- Joining Features with Labels ---")

# Inner join to keep only the claims that belong to the providers in our train set
ml_ready_df = final_joined_df.join(train_df, on="Provider", how="inner")

print("✅ Successfully joined Target Labels (is_fraud) with the main dataset.")

# Verify the final schema and distribution
ml_ready_df.select("Provider", "ClaimID", "TotalClaimCost", "is_fraud").limit(10).toPandas()

In [0]:
# Check class imbalance (How many claims are fraudulent vs normal?)
ml_ready_df.groupBy("is_fraud").count().limit(10).toPandas()

In [0]:
display(ml_ready_df)

In [0]:
# create a copy of the dataframe where "NA" strings are treated as actual NULLs
# use .when().otherwise() to replace "NA" with null for all string columns
df_with_nulls = ml_ready_df
for col in ml_ready_df.columns:
    df_with_nulls = df_with_nulls.withColumn(
        col, 
        F.when(F.col(col) == "NA", None).otherwise(F.col(col))
    )

# perform the Missing Value Evaluation on the new 'df_with_nulls'
missing_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_with_nulls.columns]
missing_pd = df_with_nulls.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ['Column_Name', 'Missing_Count']
missing_pd['Missing_Percentage(%)'] = (missing_pd['Missing_Count'] / total_rows) * 100

# Sort and Display
missing_to_show = missing_pd[missing_pd['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

if not missing_to_show.empty:
    print("Found missing values (including 'NA' replaced as Null):")
    display(missing_to_show)
else:
    print("✅ No missing values (or 'NA' strings) found.")

In [0]:
# Check row that TotalClaimCost or DeductibleRatio is Null
null_check_df = ml_ready_df.filter(
    F.col("TotalClaimCost").isNull() | F.col("DeductibleRatio").isNull()
)

display(null_check_df)

In [0]:
ml_ready_df.filter(F.col("Is_Inpatient") == 1).describe("TotalClaimCost", "DeductibleRatio").toPandas()

**Filling Nulls with Group Mean**

In [0]:
# Calculate Mean for Inpatient group using F.mean explicitly
stats = ml_ready_df.filter(F.col("Is_Inpatient") == 1).select(
    F.mean("TotalClaimCost").alias("mean_cost"),
    F.mean("DeductibleRatio").alias("mean_ratio")
).collect()[0]

mean_cost = stats["mean_cost"]
mean_ratio = stats["mean_ratio"]

print(f"Calculated Mean TotalClaimCost: {mean_cost:.2f}")
print(f"Calculated Mean DeductibleRatio: {mean_ratio:.4f}")

# Fill Nulls using the calculated means
ml_ready_df = ml_ready_df.withColumn(
    "TotalClaimCost",
    F.when((F.col("Is_Inpatient") == 1) & (F.col("TotalClaimCost").isNull()), mean_cost)
    .otherwise(F.col("TotalClaimCost"))
)

ml_ready_df = ml_ready_df.withColumn(
    "DeductibleRatio",
    F.when((F.col("Is_Inpatient") == 1) & (F.col("DeductibleRatio").isNull()), mean_ratio)
    .otherwise(F.col("DeductibleRatio"))
)

print("✅ Successfully imputed missing values with group mean.")

In [0]:
# Check row that TotalClaimCost or DeductibleRatio is Null
null_check_df = ml_ready_df.filter(
    F.col("TotalClaimCost").isNull() | F.col("DeductibleRatio").isNull()
)

display(null_check_df)

In [0]:
display(ml_ready_df)


## Saving Data to Unity Catalog (Best Practice)

In [0]:
# Saving Data to Unity Catalog (Specific Schema)

catalog_name = "workspace" 
schema_name = "insurance_claim"
table_name = "ml_ready_dataset"

full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

ml_ready_df.write.mode("overwrite").saveAsTable(full_table_path)

print(f"✅ Successfully saved dataset to Unity Catalog at: {full_table_path}")

## ML Data Preparation (Encoding & Vector Assembly)

Goal: Convert string categories to numeric indices and pack features into a vector.

In [0]:
# =============================================================================
# ML Data Preparation: Encoding & Vector Assembly using Pipeline
#=============================================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.types import StringType

print("--- Starting ML Data Preparation ---")

# Load data from Unity Catalog (Adjust the catalog name if it is not 'main')
dataset_path = "workspace.insurance_claim.ml_ready_dataset"
df = spark.table(dataset_path)

# Define columns to exclude from features (Identifiers and Target Label)
# Including IDs in the training phase can lead to data leakage and severe overfitting.
ignore_cols = [
    # ID AND basic Label
    "ClaimID", "BeneID", "Provider", "is_fraud",
    "AttendingPhysician", "OperatingPhysician", "OtherPhysician",
    
    # Collect ClmDiagnosisCode_1, DiagnosisGroupCode
    "ClmDiagnosisCode_2", "ClmDiagnosisCode_3", "ClmDiagnosisCode_4", 
    "ClmDiagnosisCode_5", "ClmDiagnosisCode_6", "ClmDiagnosisCode_7", 
    "ClmDiagnosisCode_8", "ClmDiagnosisCode_9", "ClmDiagnosisCode_10",
    "ClmProcedureCode_1", "ClmProcedureCode_2", "ClmProcedureCode_3", 
    "ClmProcedureCode_4", "ClmProcedureCode_5", "ClmProcedureCode_6",

    "ClaimStartDt", "ClaimEndDt", "AdmissionDt", "DischargeDt", "DOB", "DOD"
]

# Automatically detect and separate categorical and numeric columns based on data types
categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType) and f.name not in ignore_cols]
numeric_cols = [f.name for f in df.schema.fields if not isinstance(f.dataType, StringType) and f.name not in ignore_cols]


print(f"📌 Categorical Features ({len(categorical_cols)}):\n {categorical_cols}")
print(f"\n📌 Numeric Features ({len(numeric_cols)}):\n {numeric_cols}")

In [0]:
# =============================================================================
# Building the ML Pipeline
# =============================================================================

# Create a list of output column names
indexed_cat_cols = [f"{c}_indexed" for c in categorical_cols]

# Use inputCols (plural) and outputCols (plural)
# This creates ONLY 1 model in memory instead of multiple models!
indexer = StringIndexer(
    inputCols=categorical_cols, 
    outputCols=indexed_cat_cols, 
    handleInvalid="keep"
)

# 4. Vector Assembler
assembler_inputs = numeric_cols + indexed_cat_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs, 
    outputCol="features", 
    handleInvalid="keep"
)

# 5. Build and Run Pipeline (Now only 2 stages!)
stages = [indexer, assembler]
pipeline = Pipeline(stages=stages)

print("\n⚙️ Running Optimized ML Pipeline (Memory Friendly)...")
pipeline_model = pipeline.fit(df)
ml_prepared_df = pipeline_model.transform(df)

print("✅ Data Preparation Completed")
ml_prepared_df.select("features", "is_fraud").toPandas()

## Train-Test Split & Class Distribution Check

In [0]:
print("--- Starting Train-Test Split ---")

# Separate data into Train (80%) and Test (20%)
train_data, test_data = ml_prepared_df.randomSplit([0.8, 0.2], seed=42)

print(f"Total records: {ml_prepared_df.count():,}")
print(f"Training set: {train_data.count():,}")
print(f"Testing set: {test_data.count():,}")

In [0]:
# Check balance of data (Class Distribution) in Training Set
print("\n--- Class Distribution in Training Set ---")
train_data.groupBy("is_fraud").count().toPandas()

In [0]:
train_data.write.mode("overwrite").saveAsTable("workspace.insurance_claim.train_set")
test_data.write.mode("overwrite").saveAsTable("workspace.insurance_claim.test_set")

In [0]:
mlflow.spark.log_model(pipeline_model, "fraud_pipeline_model", dfs_tmpdir=tmp_vol_path)